# MLD cross-validation

This notebook tunes the pruned maximum-likelihood decoder used by `decoder.py` on a deterministic subset of the labelled training data. Each candidate is evaluated separately for the `X` and `Z` bases using cross-validation, and the selected configuration minimizes the combined logical error rate (LER). Runtime is reported as a secondary metric.

The folds are used only for evaluation: MLD has no trainable parameters and is not fitted on the fold data.


In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = Path("..").resolve()
REPO_ROOT = REPO_ROOT.resolve()
requirements_path = REPO_ROOT / "submission_template" / "requirements.txt"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(requirements_path)])
print("installed dependencies for:", sys.executable)


In [ ]:
import time
from itertools import product

import numpy as np
import stim

sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT / "submission_template"))

from hackathon_2026 import load_kaggle_csv, logical_error_rate, problem_info

# Reload MLD from this repository so a previously imported/stale module cannot hide
# the log-weight API used by decoder.py.
import importlib
sys.modules.pop("mld", None)
mld = importlib.import_module("mld")
if not all(hasattr(mld, name) for name in ("build_mld", "marginals_logw", "mld_logw")):
    raise ImportError(f"loaded unexpected MLD module: {getattr(mld, '__file__', None)}")
build_mld = mld.build_mld
marginals_logw = mld.marginals_logw
mld_logw = mld.mld_logw

DATA_PATH = REPO_ROOT / "data" / "train" / "train.csv"
SUBSET_PER_BASIS = 5000
N_FOLDS = 3
SEED = 42
print("repository:", REPO_ROOT)
print("data:", DATA_PATH)


## Load labelled data and MLD inputs

For each basis, the notebook loads detector events and logical labels, then extracts `H`, `L`, and `p` from the corresponding Stim circuit. A separate MLD table is built for each basis and each tested `hexp` value.


In [ ]:
def load_basis_data(data_path: str | Path, subset_per_basis: int, seed: int):
    data = load_kaggle_csv(str(data_path))
    if "observable_flips" not in data:
        raise ValueError("cross-validation requires L0-L3 labels")
    result = {}
    for offset, basis in enumerate(("X", "Z")):
        rows = np.flatnonzero(data["basis"] == basis)
        if len(rows) == 0:
            raise ValueError(f"no {basis} rows found")
        rng = np.random.default_rng(seed + offset)
        rng.shuffle(rows)
        if subset_per_basis is not None:
            rows = rows[:min(subset_per_basis, len(rows))]
        result[basis] = {
            "events": data["detector_events"][rows].astype(np.uint8),
            "labels": data["observable_flips"][rows].astype(bool),
        }
    return result

def load_problem_data(data_folder: str | Path):
    problems = {}
    for basis in ("X", "Z"):
        circuit = stim.Circuit.from_file(str(Path(data_folder) / f"{basis}_circuit.stim"))
        info = problem_info(circuit)
        problems[basis] = info
        print(f"{basis}: H={info['H'].shape}, L={info['L'].shape}, p={info['p'].shape}")
    return problems

basis_data = load_basis_data(DATA_PATH, SUBSET_PER_BASIS, SEED)
problems = load_problem_data(DATA_PATH.parent)
for basis in ("X", "Z"):
    print(basis, basis_data[basis]["events"].shape, basis_data[basis]["labels"].shape)


## Deterministic cross-validation folds

In [ ]:
def make_folds(size: int, n_folds: int, seed: int) -> list[np.ndarray]:
    if n_folds < 2 or n_folds > size:
        raise ValueError("n_folds must be at least 2 and no larger than the subset size")
    indices = np.arange(size)
    np.random.default_rng(seed).shuffle(indices)
    return [fold.copy() for fold in np.array_split(indices, n_folds)]

folds = {
    basis: make_folds(len(basis_data[basis]["events"]), N_FOLDS, SEED + i)
    for i, basis in enumerate(("X", "Z"))
}
print({basis: [len(fold) for fold in folds[basis]] for basis in ("X", "Z")})


## MLD parameter grid

Each tier is `(eps, max_states, threshold)`. The first tier always decodes every shot. A later tier re-decodes shots whose current logical marginal is between `threshold` and `1 - threshold`. The grid also tests the `hexp` heuristic and the lower-level `compact_every` performance parameter.


In [ ]:
# Tiers use the same four-field format as decoder.py:
# (direction, eps, max_states, uncertainty_threshold).
# Direction 0 uses the normal detector order; direction 1 uses the reversed order.
BASE_TIERS = ((0, 1e-4, 1024, 0.0), (1, 3e-5, 8192, 0.2))
BASE_HEXP = 0.55
BASE_COMPACT_EVERY = 32


# Tune the production-equivalent two-pass decoder tightly around the current baseline.
TWO_PASS_GRID = [
    {
        "tiers": (
            (0, first_eps, first_states, 0.0),
            (1, second_eps, second_states, threshold),
        ),
        "hexp": hexp,
        "compact_every": compact_every,
    }
    for first_eps, first_states, second_eps, second_states, threshold, hexp, compact_every in product(
        (3e-4, 1e-4),
        (1024, 1536),
        (1e-5, 3e-5),
        (4096, 8192),
        (0.15, 0.20, 0.25),
        (0.55, 0.60),
        (32,),
    )
]

# Test a small number of nearby three-pass configurations.
THREE_PASS_GRID = []#[
    #{
        #"tiers": (
            #(0, 3e-4, 512, 0.0),
            #(1, 3e-5, 4096, middle_threshold),
            #(0, last_eps, last_states, final_threshold),
        #),
        #"hexp": hexp,
        #"compact_every": compact_every,
    #}
    #for middle_threshold, last_eps, last_states, final_threshold, hexp, compact_every in product(
        #(0.10, 0.15),
        #(1e-6, 3e-6),
        #(4096, 8192),
        #(0.25, 0.35),
        #(0.50, 0.55),
        #(32,),
    #)
#]

PARAM_GRID = []
_seen = set()
for candidate in TWO_PASS_GRID + THREE_PASS_GRID:
    key = repr(candidate)
    if key not in _seen:
        _seen.add(key)
        PARAM_GRID.append(candidate)

print(f"testing {len(PARAM_GRID)} production-equivalent MLD configurations")
print("baseline:", {
    "tiers": BASE_TIERS,
    "hexp": BASE_HEXP,
    "compact_every": BASE_COMPACT_EVERY,
})


In [ ]:
TABLE_CACHE = {}

def get_tables(basis: str, hexp: float):
    """Build the same forward/reverse table pair used by decoder._tables."""
    key = (basis, float(hexp))
    if key not in TABLE_CACHE:
        info = problems[basis]
        TABLE_CACHE[key] = (
            build_mld(info["H"], info["L"], info["p"], hexp=hexp),
            build_mld(info["H"][::-1].copy(), info["L"], info["p"], hexp=hexp),
        )
    return TABLE_CACHE[key]


def decode_production_marginals(tables, detector_events, tiers, compact_every):
    """Match decoder.decode_marginals for arbitrary candidate tier parameters."""
    D = np.asarray(detector_events, dtype=bool)
    events = (np.ascontiguousarray(D), np.ascontiguousarray(D[:, ::-1]))
    logw = None
    for direction, eps, max_states, threshold in tiers:
        if logw is None:
            logw = mld_logw(
                tables[direction],
                events[direction],
                eps=eps,
                max_states=max_states,
                compact_every=compact_every,
            )
            continue
        selected = np.flatnonzero(
            (np.abs(marginals_logw(logw) - 0.5) < 0.5 - threshold).any(1)
        )
        if len(selected):
            logw[selected] = np.maximum(
                logw[selected],
                mld_logw(
                    tables[direction],
                    events[direction][selected],
                    eps=eps,
                    max_states=max_states,
                    compact_every=compact_every,
                ),
            )
    return marginals_logw(logw)


def evaluate_configuration(params: dict) -> dict:
    basis_scores = {"X": [], "Z": []}
    total_wrong = 0
    total_bits = 0
    total_seconds = 0.0
    for basis in ("X", "Z"):
        events = basis_data[basis]["events"]
        labels = basis_data[basis]["labels"]
        tables = get_tables(basis, params["hexp"])
        for fold_index, test_indices in enumerate(folds[basis]):
            started = time.perf_counter()
            marginals = decode_production_marginals(
                tables,
                events[test_indices],
                params["tiers"],
                params["compact_every"],
            )
            predictions = (marginals > 0.5).astype(np.uint8)
            elapsed = time.perf_counter() - started
            score = logical_error_rate(predictions, labels[test_indices])
            wrong = int(np.count_nonzero(predictions.astype(bool) != labels[test_indices]))
            total_wrong += wrong
            total_bits += predictions.size
            total_seconds += elapsed
            basis_scores[basis].append({
                "fold": fold_index,
                "ler": score,
                "seconds": elapsed,
            })
    return {
        "params": params,
        "combined_ler": total_wrong / total_bits,
        "seconds": total_seconds,
        "basis": basis_scores,
    }


## Run the search

The objective is the combined LER across all held-out `X` and `Z` folds. Ties are broken by total runtime. The first run may spend extra time compiling the Numba kernel.


In [ ]:
results = []
print(f"evaluating {len(PARAM_GRID)} MLD configurations")
for index, params in enumerate(PARAM_GRID, start=1):
    result = evaluate_configuration(params)
    results.append(result)
    print(
        f"{index}/{len(PARAM_GRID)} "
        f"LER={result['combined_ler']:.6f} "
        f"time={result['seconds']:.2f}s "
        f"params={params}"
    )

results.sort(key=lambda result: (result["combined_ler"], result["seconds"]))
best = results[0]
print("\nbest configuration:", best["params"])
print(f"best combined LER: {best['combined_ler']:.6f}")
print(f"best CV runtime: {best['seconds']:.2f}s")


In [ ]:
for basis in ("X", "Z"):
    print(f"\n{basis} folds")
    for fold_result in best["basis"][basis]:
        print(fold_result)

print("\nUse these values in decoder.py:")
print("MLD_TIERS =", best["params"]["tiers"])
print("MLD_COMPACT_EVERY =", best["params"]["compact_every"])
print("MLD_HEXP =", best["params"]["hexp"])
print("\nRerun the selected configuration on a fresh holdout before trusting the result.")
